In [1]:

import pandas as pd
from ytmusicapi import YTMusic
cache = {}

yt = YTMusic('../headers_auth.json')

# Get a set of videoIds for thumbs up and thumbs down
def get_playlist_track_ids(playlist_id, print_stuff=False):
    playlist_meta = yt.get_playlist(playlist_id, limit=10000)
    track_ids = set()
    for i, track in enumerate(playlist_meta['tracks']):
        if print_stuff:
            try:
                print('(%d/%d) %s - %s - %s' % (
                    i+1, len(playlist_meta['tracks']), track['artists'][0]['name'], track['album']['name'], track['title']))
            except Exception as e:
                print(e)
        track_ids.add(track['videoId'])
    return track_ids


In [ ]:
db = pd.read_csv('../../music-sources-unified/ytmusic_all_database.tsv', sep='\t', index_col=0)
db.columns

In [7]:
playlists = pd.DataFrame(yt.get_library_playlists(limit=500))
jb = playlists.loc[playlists.title.str.contains('jukebox_19')]
jb

,title,playlistId,thumbnails,count
197,jukebox_1960s,PLWptjpDqazOz9eWJ5dGbIdYq5-1APQXMs,[{'url': 'https://lh3.googleusercontent.com/mI...,695
198,jukebox_1950s,PLWptjpDqazOy7_ALT-aBV4-3mCk0xoKfj,[{'url': 'https://lh3.googleusercontent.com/FL...,772


jukebox_1960s PLWptjpDqazOz9eWJ5dGbIdYq5-1APQXMs
jukebox_1960s
jukebox_1950s PLWptjpDqazOy7_ALT-aBV4-3mCk0xoKfj
jukebox_1950s


In [124]:
# %%time
for i, row in jb.iterrows():
    print(row.title, row.playlistId)
    pl_name = row.title
    id = row.playlistId
    new_pl_name = pl_name + '_new'
    pl = yt.get_playlist(id, limit=1000)

    new_pl_ids = []
    for tk in pl['tracks']:        
        query = f"{tk['artists'][0]['name']} {tk['title']}"
        if query not in cache:
            cache[query] = yt.search(query, filter='songs')
        res = cache[query]
        TOP_N = 6
        choices = []
        # take first result, or any other results if they are in db
        for r in res[0:TOP_N]:
            if r['videoId'] in db.index:
                if vid not in cache:  
                    cache[vid]=yt.get_song(vid)
                if cache[vid]['playabilityStatus']['status'] == 'OK':            
                    choices.append((r['videoId'], db.loc[r['videoId']]['likeStatus']))
                    continue
                else:
                    print(f"db entry videoId no longer found!:\n{db.loc[r['videoId']]}")
            if not len(choices):
                choices.append((r['videoId'], None))
        
        if len(choices) == 1:
            choices_filt = choices
        else:
            rating ={'LIKE': [], 'DISLIKE': [], 'INDIFFERENT': [], 'NONE': []}
            for c in choices:
                vid, likeStatus = c                
                rating[str(likeStatus).upper()].append(c)
            choices_filt = []
            if len(rating['LIKE']):
                choices_filt = rating['LIKE']
            if not len(choices_filt)and len(rating['DISLIKE']):
                choices_filt = [rating['DISLIKE'][0]]
            if not len(choices_filt) and len(rating['INDIFFERENT']):
                choices_filt = [rating['INDIFFERENT'][0]]
            if not len(choices_filt):
                choices_filt = [rating['NONE'][0]]
        for c in choices_filt:
            vid, likeStatus = c                
            new_pl_ids.append(vid)
            if tk['likeStatus'] !='INDIFFERENT' and tk['likeStatus'] != likeStatus:
                yt.rate_song(vid, tk['likeStatus'])
                print(f"rating matched song: {query} as {tk['likeStatus']}")


    pl_res = yt.create_playlist(title=new_pl_name, video_ids=new_pl_ids, description=f"based on {tk['album']['name']} cd collection")
    print(f"{pl_name} original # entires: {len(pl['tracks'])}, queried #: {len(new_pl_ids)}, new # entries {len(pl_res)}")

jukebox_1960s PLWptjpDqazOz9eWJ5dGbIdYq5-1APQXMs
rating matched song: Del Shannon Keep Searchin' as LIKE
rating matched song: Connie Francis La Vie En Rose as LIKE
rating matched song: The Ventures Cruel Sea as LIKE
rating matched song: Frank Sinatra Hello Dolly! as LIKE
rating matched song: Brian Hyland Sealed with a Kiss as LIKE
jukebox_1960s original # entires: 695, queried #: 718, new # entries 34
jukebox_1950s PLWptjpDqazOy7_ALT-aBV4-3mCk0xoKfj
rating matched song: Pee Wee King & His Cowboys Slow Poke as LIKE
rating matched song: Bill Haley & His Comets Shake Rattle & Roll as LIKE
rating matched song: Kay Starr The Prisoner's Song as LIKE
rating matched song: Tennessee Ernie Ford Sixteen Tons as LIKE
rating matched song: Bill Hayes The Ballad of Davy Crockett as LIKE
rating matched song: Roger Williams Autumn Leaves as LIKE
rating matched song: Bill Haley & The Comets Burn That Candle as LIKE
rating matched song: Gene Vincent & Blue Caps Be Bop A Lula as LIKE
rating matched song: 